# Batching over images

The parallelism *inside* a round — one batched target call for a whole draft tree — is
orthogonal to the parallelism *across* images, and a real generation job wants both. Doing them
together is not a reshape, because speculation breaks the property that makes image batching
trivial in a standard sampler:

> **trajectories accept different prefixes and immediately fall out of step.**

After one round, trajectory 0 may sit at step 4 and trajectory 1 at step 1. Three consequences,
and they are the whole design of `specdiff/batched.py`:

1. **`sigma` becomes per row.** Rows of one verification batch belong to different steps, so
   `BatchedVerifyRequest` carries `sigmas`, a tuple, not a scalar.
2. **Live rows shrink as the round descends.** A trajectory that rejects at level 1 takes no
   part in level 2. The sampler *compacts* rather than masks, so a rule never sees a dead row and
   never needs a validity flag.
3. **Cost is a max, not a mean.** One call serves every live trajectory, so the batch advances
   at the pace of its slowest member.

In [1]:
import numpy as np

from specdiff import (
    BatchedSpeculativeSampler,
    BatchedVerifyResult,
    DelayedDriftProposal,
    DraftTree,
    IdentityProposal,
    NoiseSchedule,
    ProposalTransition,
    SpeculativeSampler,
    TargetTransition,
    Verifier,
    VerifyResult,
)
from specdiff.verifiers.rank1 import Rank1Frame

rng = np.random.default_rng(0)


In [2]:
# The same toy model as the sampler tutorial.
dim, N = 4, 20


class LinearReverseKernel(TargetTransition):
    def __init__(self, gammas):
        super().__init__()
        self.gammas = np.asarray(gammas, dtype=float)

    def means(self, indices_in_batch, states, steps):
        gamma = self.gammas[list(steps)].reshape(-1, *([1] * (states.ndim - 1)))
        return states * (1.0 - gamma)


class SqrtSchedule(NoiseSchedule):
    def __init__(self, num_steps, scale=0.4, floor=0.05):
        self.num_steps, self.scale, self.floor = int(num_steps), float(scale), float(floor)

    def sigma(self, step):
        return self.floor + self.scale * np.sqrt((self.num_steps - step) / self.num_steps)


class DeltaProbe(Verifier):
    """Exact, never accepts, records delta. See the verifier tutorial."""

    name = "delta-probe"

    def __init__(self):
        self.deltas = []

    def reset(self):
        self.deltas.clear()

    def verify(self, request):
        self.deltas.append(Rank1Frame.from_request(request).delta)
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)


target = LinearReverseKernel(np.linspace(0.05, 0.25, N))
schedule = SqrtSchedule(N)
tree = DraftTree.uniform(branching=2, lookahead=3)
batch = 8
init = rng.standard_normal((batch, dim))

## 1. A batched run

Same components as `SpeculativeSampler`, with one extra requirement: the tree must be
**level-uniform**. The proposal and the verifier need no change at all.


In [3]:
sampler = BatchedSpeculativeSampler(
    target=target,
    proposal=DelayedDriftProposal(target),   # one frozen drift per trajectory
    schedule=schedule,
    tree=tree,
    verifier=DeltaProbe(),                          # unchanged from the scalar sampler
    num_steps=N,
    check_contract=True,
    keep_trajectories=True,
)

result = sampler.sample(init, rng=rng)
print(result.summary())
print()
print("samples        ", result.samples.shape, " (batch, *state_shape)")
print("trajectories   ", result.trajectories.shape, " (batch, N + 1, *state_shape)")
print("iterations     ", len(result.rounds), " = target_calls minus the batched warm-up")
print("rounds/traj    ", result.rounds_per_trajectory)

steps=20 batch=8 target_calls=21 speedup=0.952x (isolated 1.000x, occupancy 1.00) acceptance=0.000 drafted=2080 verified=1048

samples         (8, 4)  (batch, *state_shape)
trajectories    (8, 21, 4)  (batch, N + 1, *state_shape)
iterations      20  = target_calls minus the batched warm-up
rounds/traj     (20, 20, 20, 20, 20, 20, 20, 20)


## 2. Three numbers instead of one

| property | meaning |
| --- | --- |
| `speedup` | `N / target_calls` — **wall-clock**, what you actually get for a batch |
| `mean_isolated_speedup` | mean of `N / rounds_i` — what those trajectories would each have achieved alone |
| `occupancy` | mean fraction of the batch still live per iteration |

The gap between the first two is the **straggler cost**, and it is the thing you tune batch size
against. `acceptance_rate` is pooled over every trajectory and round, so it is directly
comparable with a single-trajectory run of the same configuration.

With the probe every trajectory advances one step per round, so nothing straggles and the three
numbers are degenerate. Section 4 makes them interesting.

In [4]:
print(f"speedup                {result.speedup:.3f}x")
print(f"mean_isolated_speedup  {result.mean_isolated_speedup:.3f}x")
print(f"occupancy              {result.occupancy:.3f}")
print(f"acceptance_rate        {result.acceptance_rate:.3f}")
print()
rec = result.rounds[0]
print("BatchedRoundRecord[0]:")
for field in ("iteration", "active", "start_steps", "committed", "accepted_depth", "rejected",
              "drafted", "verified"):
    print(f"  {field:<16}{getattr(rec, field)}")

speedup                0.952x
mean_isolated_speedup  1.000x
occupancy              1.000
acceptance_rate        0.000

BatchedRoundRecord[0]:
  iteration       0
  active          (0, 1, 2, 3, 4, 5, 6, 7)
  start_steps     (0, 0, 0, 0, 0, 0, 0, 0)
  committed       (1, 1, 1, 1, 1, 1, 1, 1)
  accepted_depth  (0, 0, 0, 0, 0, 0, 0, 0)
  rejected        (True, True, True, True, True, True, True, True)
  drafted         112
  verified        56


## 3. Proposals under batching

There is nothing to choose. `ProposalTransition` takes `indices_in_batch` on every call —
`indices_in_batch[i]` is which of the `batch_size` images entry `i` belongs to — so the object
you hand `SpeculativeSampler` is the object you hand `BatchedSpeculativeSampler`. Batch size 1
is `batch_size = 1`, not a different interface.

A proposal with no per-image memory (a draft network, `IdentityProposal`) ignores the argument.
`DelayedDriftProposal` uses it: its frozen increments live in a `(batch_size, *state_shape)`
buffer, so drafting is one gather-and-add and no image can read another's drift.


In [5]:
# One instance, three images, three different roots -- and an identical state to draft from.
# If the buffer were shared, all three rows would come back equal.
proposal = DelayedDriftProposal(target)
proposal.reset(3)

roots = np.stack([r * rng.standard_normal(dim) for r in (1.0, 2.0, 3.0)])
proposal.on_round_start((0, 1, 2), (0, 0, 0), roots)

shared = np.zeros((3, dim))
out = proposal.means((0, 1, 2), shared, (0, 0, 0))
print("rows differ           ", not np.allclose(out[0], out[1]))
print("each is its own drift ", np.allclose(out, shared + (target((0, 1, 2), roots, (0, 0, 0)) - roots)))


rows differ            True
each is its own drift  True


In [6]:
# The warm-up is batched: one target call for the whole batch, not one per image.
for label, proposal in (
    ("DelayedDriftProposal", DelayedDriftProposal(target)),
    ("IdentityProposal", IdentityProposal()),
):
    s = BatchedSpeculativeSampler(
        target=target, proposal=proposal, schedule=schedule, tree=tree,
        verifier=DeltaProbe(), num_steps=N,
    )
    r = s.sample(init, rng=np.random.default_rng(5))
    print(f"{label:>22}  NFEs {r.target_calls:>3}  iterations {len(r.rounds):>3}  "
          f"rows {r.target_states_evaluated:>5}")


  DelayedDriftProposal  NFEs  21  iterations  20  rows  1048
      IdentityProposal  NFEs  20  iterations  20  rows  1040


`DelayedDriftProposal` costs one NFE more than it runs iterations: that single extra call is
the warm-up, and it covers all 8 trajectories at once rather than costing one call each.
`IdentityProposal` needs no warm-up at all (and produces a much larger `delta`).

## 4. Stragglers, occupancy, and why cost is a max

To see the effect the batched sampler is built around, we need trajectories that progress at
*different* rates. The device below is artificial but exact: the proposal is perfect for
even-numbered trajectories (`delta = 0`, so a degeneracy-accepting rule takes the whole
lookahead) and useless for odd ones (`delta` large, so they crawl one step per round).

Real runs get the same effect from ordinary randomness — an easy image accepts long prefixes, a
hard one keeps rejecting.

In [7]:
class MixedQualityProposal(ProposalTransition):
    """Oracle for even images, identity for odd ones: easy and hard trajectories together."""

    def means(self, indices_in_batch, states, steps):
        out = target.means((0,) * len(steps), states, steps)                      # uncounted: a measurement device
        hard = [i for i, b in enumerate(indices_in_batch) if b % 2 == 1]
        out[hard] = states[hard]                               # m^p(y) = y for the hard ones
        return out


class AcceptIfIdentical(Verifier):
    """Exact; accepts only where the two kernels coincide (Remark 2)."""

    name = "accept-if-identical"

    def verify(self, request):
        if Rank1Frame.from_request(request).degenerate:
            return VerifyResult(request.child(0), accepted=True, child_index=0)
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)


mixed = BatchedSpeculativeSampler(
    target=target, proposal=MixedQualityProposal(), schedule=schedule, tree=tree,
    verifier=AcceptIfIdentical(), num_steps=N, check_contract=True,
)
mix = mixed.sample(init, rng=np.random.default_rng(2))
print(mix.summary())
print()
print("rounds per trajectory  ", mix.rounds_per_trajectory, " (even images race, odd ones crawl)")
print(f"wall-clock speedup      {mix.speedup:.3f}x   <- set by the slowest member")
print(f"mean isolated speedup   {mix.mean_isolated_speedup:.3f}x   <- what they would get alone")
print(f"straggler cost          {100 * (1 - mix.speedup / mix.mean_isolated_speedup):.1f}%")
print(f"occupancy               {mix.occupancy:.3f}")

steps=20 batch=8 target_calls=20 speedup=1.000x (isolated 1.929x, occupancy 0.68) acceptance=0.500 drafted=1400 verified=700

rounds per trajectory   (7, 20, 7, 20, 7, 20, 7, 20)  (even images race, odd ones crawl)
wall-clock speedup      1.000x   <- set by the slowest member
mean isolated speedup   1.929x   <- what they would get alone
straggler cost          48.1%
occupancy               0.675


In [8]:
# Occupancy decaying as the fast trajectories finish and leave the batch.
print(f"{'iteration':>10}{'live rows':>11}{'steps done (per active image)':>32}")
for r in mix.rounds:
    if r.iteration % 3 and r.iteration != len(mix.rounds) - 1:
        continue
    print(f"{r.iteration:>10}{len(r.active):>11}   {str(r.start_steps):>29}")

 iteration  live rows   steps done (per active image)
         0          8        (0, 0, 0, 0, 0, 0, 0, 0)
         3          8        (9, 3, 9, 3, 9, 3, 9, 3)
         6          8    (18, 6, 18, 6, 18, 6, 18, 6)
         9          4                    (9, 9, 9, 9)
        12          4                (12, 12, 12, 12)
        15          4                (15, 15, 15, 15)
        18          4                (18, 18, 18, 18)
        19          4                (19, 19, 19, 19)


That is the "continuous batching" gap the README calls out: trajectories that finish early leave
the batch, so late iterations run under-full. Refilling those places with fresh trajectories
would recover it, and `indices_in_batch` is already first-class in the sampler.

## 5. What a rule sees under batching

`BatchedVerifyRequest` is `VerifyRequest` with a leading batch dimension and per-row `sigmas`.
Rows are compacted as the round descends, and `indices_in_batch` says which image each row is. The
recorder below prints one round's requests as they arrive.

In [9]:
class RequestRecorder(Verifier):
    """Passes everything through to an inner rule, logging each batched request."""

    name = "recorder"

    def __init__(self, inner):
        self.inner = inner
        self.log = []

    def reset(self):
        self.inner.reset()

    def verify(self, request):
        return self.inner.verify(request)

    def verify_batch(self, request):
        self.log.append(
            (request.info["level"], request.batch_size, request.num_children,
             tuple(request.indices_in_batch), tuple(request.steps),
             tuple(round(s, 3) for s in request.sigmas), request.info["nodes"])
        )
        return self.inner.verify_batch(request)     # the default row-wise loop


recorder = RequestRecorder(AcceptIfIdentical())
watch = BatchedSpeculativeSampler(
    target=target, proposal=MixedQualityProposal(), schedule=schedule, tree=tree,
    verifier=recorder, num_steps=N,
)
watch.sample(init, rng=np.random.default_rng(2))

print("first two iterations, level by level:")
for level, bs, K, images, steps, sigmas, nodes in recorder.log[:6]:
    print(f"  level {level}: batch={bs} K={K} images={images}")
    print(f"           steps={steps}")
    print(f"           sigmas={sigmas}")
    print(f"           nodes={nodes}")

first two iterations, level by level:
  level 1: batch=8 K=2 images=(0, 1, 2, 3, 4, 5, 6, 7)
           steps=(0, 0, 0, 0, 0, 0, 0, 0)
           sigmas=(0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45)
           nodes=(0, 0, 0, 0, 0, 0, 0, 0)
  level 2: batch=4 K=2 images=(0, 2, 4, 6)
           steps=(1, 1, 1, 1)
           sigmas=(0.44, 0.44, 0.44, 0.44)
           nodes=(1, 1, 1, 1)
  level 3: batch=4 K=2 images=(0, 2, 4, 6)
           steps=(2, 2, 2, 2)
           sigmas=(0.429, 0.429, 0.429, 0.429)
           nodes=(3, 3, 3, 3)
  level 1: batch=8 K=2 images=(0, 1, 2, 3, 4, 5, 6, 7)
           steps=(3, 1, 3, 1, 3, 1, 3, 1)
           sigmas=(0.419, 0.44, 0.419, 0.44, 0.419, 0.44, 0.419, 0.44)
           nodes=(0, 0, 0, 0, 0, 0, 0, 0)
  level 2: batch=4 K=2 images=(0, 2, 4, 6)
           steps=(4, 4, 4, 4)
           sigmas=(0.408, 0.408, 0.408, 0.408)
           nodes=(1, 1, 1, 1)
  level 3: batch=4 K=2 images=(0, 2, 4, 6)
           steps=(5, 5, 5, 5)
           sigmas=(0.396, 0

Three things to read off, and they are the three consequences from the top of this notebook:

* the **batch shrinks** from level to level as trajectories reject and drop out (compaction);
* `steps` and therefore `sigmas` **differ across rows** once trajectories fall out of step —
  broadcast them with `ops.scale_rows`, never with a scalar;
* `nodes` is a tuple, because rows sit at different nodes of the tree.
  `request.row(j)` turns it back into the scalar contract's `info["node"]`, which is why a rule
  keyed on `info["node"]` runs unchanged under both samplers.

## 6. Vectorising `verify_batch`

The default `verify_batch` loops over rows, so every rule works under batching for free. Override
it when the per-node work is worth vectorising — the `d`-dimensional projections become single
batched ops. The override must stay **row-independent**: row `j` may depend only on
`request.row(j)`.

In [10]:
class VectorisedResample(Verifier):
    """The resampling rule, done in one batched op instead of a Python loop."""

    name = "vectorised-resample"

    def verify(self, request):                       # scalar path, for reference
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)

    def verify_batch(self, request):
        ops = self.backend_for(request)              # resolves from request.children
        noise = ops.randn_stack(request.batch_size, request.target_mean[0], request.rng)
        states = request.target_mean + ops.scale_rows(noise, request.sigmas)   # per-row sigma
        n = request.batch_size
        return BatchedVerifyResult(
            states=states,
            accepted=(False,) * n,
            child_index=(None,) * n,
            proposals_examined=(0,) * n,
        )


def run(verifier, seed=11):
    s = BatchedSpeculativeSampler(
        target=target, proposal=DelayedDriftProposal(target), schedule=schedule,
        tree=tree, verifier=verifier, num_steps=N, check_contract=True,
    )
    return s.sample(init, rng=np.random.default_rng(seed))


loop, vec = run(DeltaProbe()), run(VectorisedResample())
print("row-wise loop :", loop.summary())
print("vectorised    :", vec.summary())
print("same accounting:", [r.committed for r in loop.rounds] == [r.committed for r in vec.rounds])

row-wise loop : steps=20 batch=8 target_calls=21 speedup=0.952x (isolated 1.000x, occupancy 1.00) acceptance=0.000 drafted=2080 verified=1048
vectorised    : steps=20 batch=8 target_calls=21 speedup=0.952x (isolated 1.000x, occupancy 1.00) acceptance=0.000 drafted=2080 verified=1048
same accounting: True


Identical accounting; the states themselves differ because the two consume the shared RNG stream
in a different order. That is expected, and it is why the repo proves an override is an
optimisation rather than a behaviour change with a dedicated test —
`tests/test_batched.py::test_vectorised_verify_batch_agrees_with_the_row_loop`.

`check_contract=True` applies identical per-row checks on both paths, so a rule the scalar sampler
rejects is rejected under batching too, with the same message plus a row number:

In [11]:
class BrokenRule(Verifier):
    name = "broken"

    def verify(self, request):
        return VerifyResult(request.child(0) + 1e-6, accepted=True, child_index=0)


try:
    run(BrokenRule())
except ValueError as exc:
    print("ValueError ->", str(exc)[:150], "...")

ValueError -> broken row 5 reported accepted=True but the returned state is not child 0. An accepted state must be the drafted state itself, otherwise the sampler d ...


## 7. The level-uniform requirement

Every node at a given depth must have the same number of children, or a level's candidates
cannot form a rectangular `(batch, K, *state_shape)` array. `uniform` and `from_widths` qualify;
an arbitrary pruned tree does not, and the sampler refuses it at construction.

In [12]:
pruned = DraftTree([-1, 0, 0, 1])        # node 1 has a child, node 2 does not
print(pruned, " level-uniform:", pruned.is_level_uniform())

try:
    BatchedSpeculativeSampler(
        target=target, proposal=DelayedDriftProposal(target), schedule=schedule,
        tree=pruned, verifier=DeltaProbe(), num_steps=N,
    )
except ValueError as exc:
    print("ValueError ->", str(exc).splitlines()[0])

print()
for t in (DraftTree.uniform(3, 2), DraftTree.from_widths([3, 1, 2]), pruned):
    print(f"{str(t):>44}  level-uniform {str(t.is_level_uniform()):>5}  uniform {t.is_uniform()}")

DraftTree(irregular, K=2, L=2, B=3, |I|=2)  level-uniform: False
ValueError -> DraftTree(irregular, K=2, L=2, B=3, |I|=2) is not level-uniform, so its candidates cannot form a rectangular batch. Use DraftTree.uniform / from_widths, or the single-trajectory sampler.

   DraftTree(uniform, K=3, L=2, B=12, |I|=4)  level-uniform  True  uniform True
 DraftTree(irregular, K=3, L=3, B=12, |I|=7)  level-uniform  True  uniform False
  DraftTree(irregular, K=2, L=2, B=3, |I|=2)  level-uniform False  uniform False


## 8. The two samplers agree

`sampler.py` and `batched.py` implement the same three phases, and a batch of 1 must reproduce
the single-trajectory accounting exactly. (The repo asserts this in `tests/test_batched.py`.)

In [13]:
one = init[:1]
scalar = SpeculativeSampler(
    target=target, proposal=DelayedDriftProposal(target), schedule=schedule, tree=tree,
    verifier=DeltaProbe(), num_steps=N,
).sample(one[0], rng=np.random.default_rng(4))

batched_one = BatchedSpeculativeSampler(
    target=target, proposal=DelayedDriftProposal(target), schedule=schedule, tree=tree,
    verifier=DeltaProbe(), num_steps=N,
).sample(one, rng=np.random.default_rng(4))

print(f"{'':>14}{'NFEs':>6}{'rows':>7}{'drafts':>8}{'speedup':>10}")
print(f"{'scalar':>14}{scalar.target_calls:>6}{scalar.target_states_evaluated:>7}"
      f"{scalar.drafted_states:>8}{scalar.speedup:>9.3f}x")
print(f"{'batch of 1':>14}{batched_one.target_calls:>6}{batched_one.target_states_evaluated:>7}"
      f"{batched_one.drafted_states:>8}{batched_one.speedup:>9.3f}x")

                NFEs   rows  drafts   speedup
        scalar    21    131     260    0.952x
    batch of 1    21    131     260    0.952x


Trajectories in a batch **share one RNG stream**, so a given trajectory is not bit-reproducible
across different batch sizes. Its law is unaffected — only the interleaving of the draws changes.

## 9. Choosing a batch size

The straggler cost is arithmetic: given a per-level acceptance `alpha`, each round a trajectory
advances `min(Geom(alpha), L - 1) + 1` steps, and the batch pays the **max** over its live
members. No sampler needed.

In [14]:
def plan_batch(alpha, lookahead, num_steps, batch_size, trials=200, seed=0):
    """(batched speedup, isolated speedup) for a given acceptance probability."""
    r = np.random.default_rng(seed)
    batched, isolated = [], []
    for _ in range(trials):
        rounds = np.zeros(batch_size, dtype=int)
        steps = np.zeros(batch_size, dtype=int)
        while (steps < num_steps).any():
            live = steps < num_steps
            run_len = r.geometric(1.0 - alpha, size=batch_size) - 1
            advance = np.minimum(np.minimum(run_len, lookahead - 1) + 1, num_steps - steps)
            steps = np.where(live, steps + advance, steps)
            rounds += live
        batched.append(num_steps / rounds.max())
        isolated.append(np.mean(num_steps / rounds))
    return float(np.mean(batched)), float(np.mean(isolated))


alpha, L, steps = 0.84, 3, 100
print(f"alpha={alpha}  L={L}  N={steps}")
print(f"{'batch':>7}{'batched':>11}{'isolated':>11}{'straggler cost':>17}")
for b in (1, 2, 4, 8, 16, 64):
    bs, iso = plan_batch(alpha, L, steps, b)
    print(f"{b:>7}{bs:>10.2f}x{iso:>10.2f}x{100 * (1 - bs / iso):>16.1f}%")

alpha=0.84  L=3  N=100
  batch    batched   isolated   straggler cost
      1      2.52x      2.52x             0.0%
      2      2.47x      2.53x             2.3%
      4      2.41x      2.52x             4.7%


      8      2.36x      2.53x             6.6%


     16      2.31x      2.52x             8.5%


     64      2.25x      2.53x            11.0%


The cost grows with batch size (the max over more geometric draws is larger) and it is not
negligible: ~12% at a batch of 16 for these parameters. Deeper trees make it worse, because a
round's advance is more variable.

## Recap

* Same three phases as the scalar sampler; the batch dimension changes three things: **per-row
  `sigma`**, **compaction** of live rows, and **cost as a max**.
* Proposals need no change: `ProposalTransition` carries `indices_in_batch` at every batch
  size, and `DelayedDriftProposal` keys its drift buffer on it, so images cannot contaminate
  each other.
* Rules need no change; `verify_batch` defaults to a row loop, and an override must stay
  row-independent.
* Read `speedup` for wall clock, `mean_isolated_speedup` for the per-image number, and
  `occupancy` for how much of the batch was doing useful work.

**Next:** [`backends_tutorial.ipynb`](backends_tutorial.ipynb) — the array shim, and running all
of this on torch.